# Prepare training tensor

Create a tensor with columns [x_norm, y_norm, flux_x_scaled, flux_y_scaled]

In [6]:
import numpy as np
import pandas as pd
import xarray as xr
import torch
import matplotlib.pyplot as plt

In [7]:
# import region bounds
from regions import ROSS_BOUNDS

# assign region bounds
x_min = ROSS_BOUNDS["x_min"]
x_max = ROSS_BOUNDS["x_max"]
y_min = ROSS_BOUNDS["y_min"]
y_max = ROSS_BOUNDS["y_max"]

In [8]:
# Load thickness points
# NOTE: Now we removed outliers (4 steps)
thickness = pd.read_csv("data/bedmap123_ross_onshelf_3M.csv")

In [9]:
thickness.describe()

,lon,lat,x,y,s,t,b,t_fa,x_norm,y_norm
count,3.239963e+06,3.239963e+06,3.239963e+06,3.239963e+06,3.239963e+06,3.239963e+06,3.239963e+06,3.239963e+06,3.239963e+06,3.239963e+06
mean,-2.248050e+01,-8.056348e+01,-6.485234e+04,-9.958565e+05,-1.309636e+02,4.335179e+02,-5.545172e+02,4.235179e+02,5.351477e-01,4.041435e-01
std,1.665693e+02,1.807049e+00,2.473085e+05,1.955685e+05,1.173392e+03,1.644394e+02,1.123187e+03,1.644394e+02,2.473085e-01,1.955685e-01
min,-1.838281e+02,-8.535144e+01,-5.854745e+05,-1.360496e+06,-9.999000e+03,0.000000e+00,-9.999000e+03,-1.000000e+01,1.452545e-02,3.950352e-02
25%,-1.677512e+02,-8.197557e+01,-2.385320e+05,-1.146891e+06,-4.872615e+00,3.312308e+02,-5.059273e+02,3.212308e+02,3.614680e-01,2.531094e-01
50%,-1.549330e+02,-8.024463e+01,-6.078004e+04,-1.010261e+06,5.045533e+00,3.834600e+02,-3.804200e+02,3.734600e+02,5.392200e-01,3.897391e-01
75%,1.695937e+02,-7.916884e+01,1.391966e+05,-8.502413e+05,1.955000e+01,5.171889e+02,-3.324044e+02,5.071889e+02,7.391966e-01,5.497587e-01
max,1.949701e+02,-7.739090e+01,3.894639e+05,-4.497456e+05,1.050800e+04,2.164640e+03,9.999000e+03,2.154640e+03,9.894639e-01,9.502544e-01


In [10]:
ice_vel = xr.load_dataset("/home/kim/data/nsidc/antarctic_ice_vel_phase_map_v01.nc")

velocity = ice_vel.sel(
    x = slice(x_min, x_max), 
    y = slice(y_max, y_min))

In [11]:
# Interpolate
vel_at_points = velocity[["VX", "VY"]].interp(
    x = ("points", thickness["x"].values),
    y = ("points", thickness["y"].values),
    method = "linear",
)

# Add to dataset
thickness["VX"] = vel_at_points["VX"].values
thickness["VY"] = vel_at_points["VY"].values

print(thickness.shape)

(3239963, 14)


In [12]:
thickness["VX"] = vel_at_points["VX"].values
thickness["VY"] = vel_at_points["VY"].values
# thickness["speed"] = np.hypot(thickness["VX"], thickness["VY"])

# Multiply velocity components by thickness (t_fa) to get flux components
thickness["VX_flux"] = thickness["VX"] * thickness["t_fa"]
thickness["VY_flux"] = thickness["VY"] * thickness["t_fa"]

# If you also want total flux magnitude
thickness["flux_mag"] = np.hypot(thickness["VX_flux"], thickness["VY_flux"])

# Optionally drop rows with NaNs if you want a clean dataset
# Check for NaNs
print(thickness[["VX_flux", "VY_flux", "flux_mag"]].isna().sum())
# Remove
thickness = thickness.dropna(subset = ["VX_flux", "VY_flux", "flux_mag"])
print(thickness.shape)

VX_flux     31874
VY_flux     31874
flux_mag    31874
dtype: int64
(3208089, 17)


# Visualise

In [13]:
modis_ross = torch.load("data/modis/moa125_2014_hp1_v01_ross_with_grid_no_ocean.pt", weights_only = False)

In [ ]:
SUBSAMPLE_RATE = 100

fig, ax = plt.subplots(figsize = (8, 6))

# 1) MODIS as background (draw first)
ax.pcolormesh(
    modis_ross[0],
    modis_ross[1], 
    modis_ross[2],
    cmap = "gray",
    # softer greys
    vmin = -30_000, 
    vmax = 30_000,
    # as less saturated background
    alpha = 0.4,
    zorder = 0,
)

# 2) SMB on top (draw second)
ax.scatter(
    thickness["x"][::SUBSAMPLE_RATE], 
    thickness["y"][::SUBSAMPLE_RATE], 
    c = thickness["flux_mag"][::SUBSAMPLE_RATE], 
    s = 0.6, 
    alpha = 0.8, 
    cmap = "turbo",
    vmin = 0,
    vmax = 400_000 
    )

ax.set_aspect("equal")
ax.set_axis_off()

fig.savefig("figures/ice_flux_ross.png", dpi = 300, bbox_inches = "tight", pad_inches = 0)
plt.show()

# cmap

In [ ]:
cmap = "turbo"
dpi = 300

# 1 x N gradient
grad = np.linspace(0, 1, 2048)[None, :]

fig = plt.figure(figsize = (6, 0.6), dpi = dpi)
ax = fig.add_axes([0, 0, 1, 1]) 
ax.imshow(grad, aspect = "auto", cmap = cmap)
ax.set_axis_off()

fig.savefig("figures/cmap/jet_bar.png", dpi = dpi, transparent = True, bbox_inches = "tight", pad_inches = 0)
plt.close(fig)

# Hist

In [ ]:
plt.figure(figsize = (8, 6))
plt.hist(thickness["flux_mag"], bins = 50, edgecolor="white")
plt.xlabel("Flux magnitude")
plt.ylabel("Count")
plt.title("Histogram of Ice Flux Magnitude")
plt.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Print variance after normalisation
print(np.var(thickness["VX_flux"] / 100_000))
print(np.var(thickness["VY_flux"] / 100_000))

In [ ]:
# Normalize into [0, 1]
thickness["x_norm"] = (thickness["x"] - x_min) / (x_max - x_min)
thickness["y_norm"] = (thickness["y"] - y_min) / (y_max - y_min)

# Build tensor with x_norm, y_norm, scaled flux
flux_tensor = torch.tensor(
    np.stack([
        thickness["x_norm"].values,
        thickness["y_norm"].values,
        thickness["VX_flux"].values / 100_000,
        thickness["VY_flux"].values / 100_000,
    ], axis = 1),
    dtype = torch.float32
)

print(flux_tensor.shape)
print(flux_tensor[:5])  # preview first rows

In [ ]:
import matplotlib.pyplot as plt

# Extract the last two columns
vx_flux_scaled = flux_tensor[:, 2].numpy()
vy_flux_scaled = flux_tensor[:, 3].numpy()

# Plot histograms
fig, axes = plt.subplots(1, 2, figsize = (12, 5), sharey = True)

axes[0].hist(vx_flux_scaled, bins = 50, edgecolor = "white")
axes[0].set_title("VX_flux / 100_000")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Count")
axes[0].grid(alpha = 0.3)

axes[1].hist(vy_flux_scaled, bins = 50, edgecolor = "white", color = "orange")
axes[1].set_title("VY_flux / 100000")
axes[1].set_xlabel("Value")
axes[1].grid(alpha = 0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 500 * 200
# 425 meter ice * 438 meter per year
425 * 438

In [ ]:
torch.save(flux_tensor, "data/flux_train_tensor.pt")